## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import json
import re
import math

from haystack import Pipeline, component
from hayhooks import BasePipelineWrapper
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

from openai import OpenAI
from openai.types.create_embedding_response import CreateEmbeddingResponse

In [ ]:
from sem_rag.config.settings import *
from sem_rag.retrieval.bm25 import BM25Retriever
from sem_rag.utils.image import encode_image

In [ ]:
INSTRUCTION = "Represent the user's input."

TOP_K = 20

In [ ]:
QUERIES = [
    "SEM image showing spherical nanoparticles",
]

In [ ]:
client = OpenAI(base_url=VLLM.API_URL, api_key=VLLM.API_KEY)

In [ ]:
def get_document_store() -> QdrantDocumentStore:
    """
    Returns a QdrantDocumentStore pointing at the shared, persistent, in-cluster Qdrant instance.
    """
    return QdrantDocumentStore(
        url=Qdrant.URL,
        port=80,
        index=Qdrant.COLLECTION_QWEN,
        embedding_dim=Qdrant.EMBEDDING_DIM_QWEN,
        timeout=60,
    )

In [ ]:
qdrant = get_document_store()

# BM25

In [ ]:
bm25_retriever = BM25Retriever(Evaluation.BM25_DIR)

In [ ]:
bm25_results = bm25_retriever.search(
    QUERIES[0],
    top_k=TOP_K,
)

for result in bm25_results:
    print(result["image_id"], result["score"], result["description"])
    print("_"*100)

# RAG System

In [ ]:
METADATA_SCHEMA = {
    # __________ Numeric fields __________
    "pixel_size_nm": {
        "type": "numeric",
        "description": (
            "Physical spatial sampling represented by one image pixel. "
            "It expresses the real-space scale of the SEM image and allows "
            "images acquired at different spatial scales to be compared."
        ),
        "dictionary_terms": [
            "pixel size",
            "image pixel size",
            "pixel dimension",
            "nm per pixel",
            "nm/pixel",
            "nanometers per pixel",
            "nanometres per pixel",
            "um per pixel",
            "µm per pixel",
            "micrometers per pixel",
            "micrometres per pixel",
            "spatial sampling",
            "pixel sampling",
            "sampling resolution",
        ],
        "unit": "nm/pixel",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "magnification_x": {
        "type": "numeric",
        "description": (
            "Nominal SEM magnification used when the image was acquired."
        ),
        "dictionary_terms": [
            "magnification",
            "mag",
            "magnified",
            "magnification factor",
            "times magnification",
            "x magnification",
            "kx",
            "KX",
            "zoom level",
        ],
        "unit": "x",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "accelerating_voltage_kv": {
        "type": "numeric",
        "description": (
            "Electron beam accelerating voltage, commonly reported as EHT. "
            "It influences electron penetration, interaction volume, "
            "surface sensitivity, charging behaviour and signal generation."
        ),
        "dictionary_terms": [
            "accelerating voltage",
            "acceleration voltage",
            "EHT",
            "beam voltage",
            "electron beam voltage",
            "electron voltage",
            "high tension",
            "high voltage",
            "operating voltage",
            "kV",
            "kilovolts",
            "kilovolt",
        ],
        "unit": "kV",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "working_distance_mm": {
        "type": "numeric",
        "description": (
            "SEM working distance associated with the specimen-to-objective "
            "imaging geometry. It influences imaging conditions such as depth "
            "of field, resolution and detector geometry."
        ),
        "dictionary_terms": [
            "working distance",
            "WD",
            "work distance",
            "working dist",
            "working-distance",
            "focus distance",
            "sample distance",
            "specimen distance",
        ],
        "unit": "mm",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "brightness_percent": {
        "type": "numeric",
        "description": (
            "Instrument brightness setting applied during acquisition. "
            "It controls the displayed or saved signal intensity level."
        ),
        "dictionary_terms": [
            "brightness",
            "brightness level",
            "image brightness",
            "brightness setting",
            "brightness percentage",
            "bright",
            "intensity level",
        ],
        "unit": "%",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "contrast_percent": {
        "type": "numeric",
        "description": (
            "Instrument contrast setting applied during acquisition. "
            "It controls the separation and dynamic range of intensities "
            "represented in the SEM image."
        ),
        "dictionary_terms": [
            "contrast",
            "contrast level",
            "image contrast",
            "contrast setting",
            "contrast percentage",
            "dynamic range",
            "intensity contrast",
        ],
        "unit": "%",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "scan_speed_setting": {
        "type": "numeric",
        "description": (
            "Instrument-specific scan speed setting controlling how rapidly "
            "the electron beam scans the image. It is an ordinal acquisition "
            "setting rather than a physical time unless separately calibrated."
        ),
        "dictionary_terms": [
            "scan speed",
            "scan rate",
            "scanrate",
            "scanning speed",
            "scanning rate",
            "scan speed setting",
            "scan setting",
            "acquisition speed",
            "raster speed",
        ],
        "unit": "setting",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "frame_average_count": {
        "type": "numeric",
        "description": (
            "Number of image frames averaged together during acquisition "
            "to reduce random noise and improve signal-to-noise ratio."
        ),
        "dictionary_terms": [
            "frame averaging",
            "frame average",
            "frame average count",
            "frame averaging count",
            "frames averaged",
            "number of frames averaged",
            "frames to average",
            "average frames",
            "frame count",
        ],
        "unit": "count",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    "line_average_count": {
        "type": "numeric",
        "description": (
            "Number of repeated scan lines averaged during acquisition "
            "to reduce noise and improve signal stability."
        ),
        "dictionary_terms": [
            "line averaging",
            "line average",
            "line average count",
            "line averaging count",
            "lines averaged",
            "number of lines averaged",
            "lines to average",
            "average lines",
        ],
        "unit": "count",
        "filter": {
            "type": "range",
            "tolerance": 0.10,
            "tolerance_type": "relative",
        },
    },

    # __________ Categorical fields __________
    "category": {
        "type": "categorical",
        "description": (
            "Human-assigned SEM morphology category from the annotated "
            "dataset described by Aversa et al. The categories were defined "
            "mainly from the dominant shape, structure and dimensionality "
            "of the imaged object."
        ),
        "values": {
            "Porous_Sponge": {
                "description": (
                    "SEM images assigned to the porous sponge category, "
                    "characterized by a porous or sponge-like morphology."
                ),
                "dictionary_terms": [
                    "porous sponge",
                    "porous sponges",
                    "sponge-like",
                    "sponge like",
                ],
            },
            "Patterned_surface": {
                "description": (
                    "SEM images of surfaces containing organized, repeated "
                    "or lithographically defined patterns or structures."
                ),
                "dictionary_terms": [
                    "patterned surface",
                    "patterned surfaces",
                    "surface pattern",
                    "surface patterns",
                    "patterned",
                ],
            },
            "Particles": {
                "description": (
                    "Zero-dimensional particle-like objects that may be "
                    "individually dispersed or grouped into clustered structures."
                ),
                "dictionary_terms": [
                    "particle",
                    "particles",
                    "nanoparticle",
                    "nanoparticles",
                    "nano particle",
                    "nano particles",
                ],
            },
            "Films_Coated_Surface": {
                "description": (
                    "Two-dimensional films or coatings on a surface, including "
                    "surfaces with different degrees of roughness or particle coverage."
                ),
                "dictionary_terms": [
                    "film",
                    "films",
                    "coated surface",
                    "coated surfaces",
                    "coating",
                    "coatings",
                    "thin film",
                    "thin films",
                ],
            },
            "Powder": {
                "description": (
                    "SEM images assigned to the powder category in the annotated dataset."
                ),
                "dictionary_terms": [
                    "powder",
                    "powders",
                    "powder sample",
                    "powder samples",
                ],
            },
            "Tips": {
                "description": (
                    "Tip-like structures, including images commonly associated "
                    "with atomic force microscopy probe tips."
                ),
                "dictionary_terms": [
                    "tip",
                    "tips",
                    "AFM tip",
                    "AFM tips",
                    "probe tip",
                    "probe tips",
                ],
            },
            "Nanowires": {
                "description": (
                    "One-dimensional nanowire structures. In the dataset hierarchy, "
                    "nanowires can include individual, few, entangled or forest-like arrangements."
                ),
                "dictionary_terms": [
                    "nanowire",
                    "nanowires",
                    "nano wire",
                    "nano wires",
                ],
            },
            "Biological": {
                "description": (
                    "SEM images of biological samples grouped into the Biological "
                    "category of the annotated dataset."
                ),
                "dictionary_terms": [
                    "biological",
                    "biological sample",
                    "biological samples",
                    "bio sample",
                    "bio samples",
                ],
            },
            "MEMS_devices_and_electrodes": {
                "description": (
                    "Microelectromechanical devices, electrodes and related "
                    "microfabricated structures, typically associated with lithographic fabrication."
                ),
                "dictionary_terms": [
                    "MEMS",
                    "MEMS device",
                    "MEMS devices",
                    "electrode",
                    "electrodes",
                    "microelectromechanical",
                    "microelectromechanical system",
                    "microelectromechanical systems",
                    "microdevice",
                    "microdevices",
                ],
            },
            "Fibres": {
                "description": (
                    "One-dimensional fibrous structures assigned to the dedicated "
                    "Fibres category of the annotated dataset."
                ),
                "dictionary_terms": [
                    "fibre",
                    "fibres",
                    "fiber",
                    "fibers",
                    "fibrous",
                ],
            },
        },
    },

    "detector": {
        "type": "categorical",
        "description": (
            "Detector or signal channel used to form the SEM image. "
            "Detector choice affects the type of electron signal recorded "
            "and therefore the appearance and scientific information represented in the image."
        ),
        "values": {
            "InLens": {
                "description": (
                    "In-lens detector signal used for SEM image formation. "
                    "The instrument described in the dataset paper includes "
                    "a high-efficiency in-lens detector."
                ),
                "dictionary_terms": [
                    "InLens",
                    "in lens",
                    "in-lens",
                    "inlens",
                    "in lens detector",
                    "in-lens detector",
                ],
            },
            "SE2": {
                "description": (
                    "Secondary-electron detector channel labelled SE2 "
                    "in the instrument metadata."
                ),
                "dictionary_terms": [
                    "SE2",
                    "se2",
                    "secondary electron",
                    "secondary electrons",
                    "secondary electron detector",
                    "SE detector",
                ],
            },
            "HDAsB": {
                "description": (
                    "Backscattered-electron detector channel labelled HDAsB "
                    "in the instrument metadata."
                ),
                "dictionary_terms": [
                    "HDAsB",
                    "hdasb",
                    "backscatter detector",
                    "backscattered electron detector",
                    "backscattered electrons",
                    "BSE detector",
                    "BSE",
                ],
            },
        },
    },
}

# __________ METADATA SCHEMA HELPERS __________
def build_metadata_schema_description() -> str:
    """
    Convert the internal metadata schema into JSON for the metadata
    extraction LLM.

    The schema contains all information the LLM needs to map natural
    language to canonical metadata fields and values.
    """
    return json.dumps(METADATA_SCHEMA, indent=2, ensure_ascii=False)

METADATA_SCHEMA_DESCRIPTION = build_metadata_schema_description()

In [ ]:
@component
class QueryEmbedder():
    """
    Generate a Qwen3-VL embedding from either text or an image.

    If an image is provided, the image is embedded directly.
    Otherwise the textual query is embedded.
    """
    def __init__(self, model: str, api_base_url: str, api_key: str):
        self.model = model
        self.client = OpenAI(base_url=api_base_url, api_key=api_key)

    @component.output_types(embedding=list[float])
    def run(self, text: str = "", image_data_urls: list[str] | None = None):
        if image_data_urls:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [
                        {"type": "image_url", "image_url": {"url": image_data_urls[0]}},
                        {"type": "text", "text": ""},
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]
        else:
            messages = [
                {"role": "system", "content": [{"type": "text", "text": INSTRUCTION,}]},
                {"role": "user", "content": [{"type": "text", "text": text}]},
                {"role": "assistant", "content": [{"type": "text", "text": ""}]},
            ]

        response: CreateEmbeddingResponse = self.client.post(
            "/embeddings",
            cast_to=CreateEmbeddingResponse,
            body={
                "messages": messages,
                "model": self.model,
                "encoding_format": "float",
                "continue_final_message": True,
                "add_special_tokens": True,
            },
        )

        return {"embedding": response.data[0].embedding}

In [ ]:
class PipelineWrapper(BasePipelineWrapper):
    def setup(self) -> None:
        document_store = get_document_store()

        # __________ RETRIEVAL PIPELINE __________
        self.retrieval_pipeline = Pipeline()

        self.retrieval_pipeline.add_component("query_embedder", QueryEmbedder(model=VLLM.EMBEDDING_MODEL, api_base_url=VLLM.API_URL, api_key=VLLM.API_KEY))
        self.retrieval_pipeline.add_component("retriever", QdrantEmbeddingRetriever(document_store=document_store, top_k=TOP_K))

        self.retrieval_pipeline.connect("query_embedder.embedding", "retriever")

    def retrieve_documents(self, question: str, images: list[str]):
        """
        Execute the retrieval stage synchronously.

        This is intentionally separated from generation so that we can
        detect zero-result retrievals before invoking the LLM.
        """
        retrieval_run_args = {
            "query_embedder": {
                "text": question,
                "image_data_urls": images,
            },
        }

        response = self.retrieval_pipeline.run(retrieval_run_args)

        documents = (response.get("retriever", {}).get("documents", []))

        return documents

In [ ]:
qwen_retriever = PipelineWrapper()
qwen_retriever.setup()

In [ ]:
qwen_results = qwen_retriever.retrieve_documents(
    QUERIES[0],
    [],
)

for result in qwen_results:
    print(result.id, result.content)

# CLIP

# LLM as a Judge

## Qwen3.5 122B A10B

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator of image retrieval systems for
Scanning Electron Microscopy (SEM) images.

Your task is to judge how relevant a candidate SEM image is
to a user's retrieval query.

The candidate IMAGE is the PRIMARY source of evidence for your judgment.

The candidate DESCRIPTION is SECONDARY supporting information.
Use it only to clarify or supplement what is visible in the image.

If the description conflicts with what is visible in the image,
trust the image.

Do not consider information present only in the description as
evidence unless it can reasonably be supported by the image.

Evaluate the candidate independently from other candidates.

Consider:
- morphology
- particle shape
- particle size when reliably inferable
- structure
- texture
- spatial arrangement
- visible patterns
- other visual characteristics explicitly requested by the query

Do not reward an image merely because it is an SEM image.

Do not invent information that is not visible in the image.

If the query specifies a numerical property such as particle size,
do not assume that value from visual appearance alone unless the
image provides sufficient evidence, such as a scale bar.

Use this relevance scale:

0 = Not relevant
1 = Slightly relevant
2 = Moderately relevant
3 = Highly relevant
4 = Extremely relevant / excellent match

Return ONLY valid JSON in exactly this format:

{
  "relevance": <integer from 0 to 4>
}

Do not include markdown.
Do not include explanations.
Do not include any additional fields.
"""

In [ ]:
def judge_candidate(
    query: str,
    image_url: str,
    description: str | None = None,
) -> dict:

    user_content = [
        {
            "type": "text",
            "text": f"""
            Query:
            {query}

            The following image is the candidate being evaluated.
            The IMAGE is the primary evidence for your judgment.
            """,
        },
        {
            "type": "image_url",
            "image_url": {
                "url": image_url
            },
        },
    ]

    if description:
        user_content.append(
            {
                "type": "text",
                "text": f"""
                Supporting candidate description:
                {description}

                Use this description only as secondary supporting information.
                Base your relevance judgment primarily on the image.
                """,
            }
        )

    response = client.chat.completions.create(
        model=VLLM.JUDGE_MODEL,
        messages=[
            {
                "role": "system",
                "content": JUDGE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": user_content,
            },
        ],
        temperature=0.0,
    )

    raw_output = response.choices[0].message.content.strip()

    try:
        result = json.loads(raw_output)

    except json.JSONDecodeError:
        match = re.search(
            r"\{.*\}",
            raw_output,
            re.DOTALL,
        )

        if not match:
            raise ValueError(
                f"Could not parse judge response:\n{raw_output}"
            )

        result = json.loads(match.group(0))

    relevance = result.get("relevance")

    if relevance not in [0, 1, 2, 3, 4]:
        raise ValueError(
            f"Invalid relevance score: {relevance}\n"
            f"Raw response: {raw_output}"
        )

    return {
        "relevance": relevance,
        # "reason": result.get("reason", ""),
        # "raw_response": raw_output,
    }

In [ ]:
image_url = encode_image(SEMDataset.CROPS / "38_a6a840c41ec3a131a5d6de121831afb7797e181926bfdfed50b62b3335899f52_512.npy")

query = "SEM image showing spherical particles around 5 micrometers"

result = judge_candidate(
    query=query,
    image_url=image_url,
    description=None,
)

print(result)

## Qwen3 VL Reranker (Not Working)

In [ ]:
RERANKER_INSTRUCTION = (
    "Evaluate the relevance of the candidate SEM image to the user's query. "
    "Consider the visual content of the image as the primary evidence. "
    "The accompanying description is supplementary information."
)

In [ ]:
from typing import Any

client = OpenAI(
    base_url=VLLM.API_URL,
    api_key=VLLM.API_KEY,
)

# query = "SEM image showing spherical particles around 5 micrometers"

# image_url = encode_image(SEMDataset.CROPS / "38_a6a840c41ec3a131a5d6de121831afb7797e181926bfdfed50b62b3335899f52_512.npy")

# response = client.post(
#     "/score",
#     cast_to=Any,
#     body={
#         "model": "Qwen/Qwen3-VL-Reranker-8B",
#         "queries": [
#             query,
#         ],
#         "documents": [
#             {
#                 "content": [
#                     {
#                         "type": "text",
#                         "text": (
#                             "SEM image showing spherical particles "
#                             "approximately 4 to 6 micrometers in diameter."
#                         ),
#                     },
#                     {
#                         "type": "image_url",
#                         "image_url": {
#                             "url": image_url,
#                         },
#                     },
#                 ],
#             },
#         ],
#     },
# )

# print(response)


response = client.post(
    "/rerank",
    body={
        "model": "Qwen/Qwen3-VL-Reranker-8B",
        "query": "Renewable energy solutions",
        "documents": [
            "Solar power adoption surged by 30% in 2024.",
            "Wind energy faces challenges in urban areas.",
            "Hydrogen fuel cells offer zero-emission transportation.",
        ],
    },
    cast_to=dict,
)

print(response)

In [ ]:
import requests

response = requests.get(
    f"{VLLM.API_URL}/models",
    headers={
        "Authorization": f"Bearer {VLLM.API_KEY}"
    }
)

print(response.status_code)
print(response.text)

In [ ]:
for endpoint in [
    "/score",
    "/rerank",
    "/rerank/",
    "/v1/score",
    "/v1/rerank",
    "/embeddings",
    "/chat/completions"
]:
    try:
        r = requests.get(
            f"{VLLM.API_URL.rstrip('/')}{endpoint}",
            headers={
                "Authorization": f"Bearer {VLLM.API_KEY}"
            },
        )
        print(f"{VLLM.API_URL.rstrip('/')}{endpoint}", r.status_code, r.text[:200])
    except Exception as e:
        print(endpoint, e)

# Results

In [ ]:
def evaluate_queries(query, bm25_results, qwen_results):
    evaluation = []

    # 1. Get the rankings from both systems
    bm25_ranking = [result["image_id"] for result in bm25_results]
    qwen_ranking = [result.id for result in qwen_results]

    # 2. Create the union of the two rankings (set to remove duplicates)
    pool = set(bm25_ranking) | set(qwen_ranking)

    # 3. Judge every unique image only once
    relevance = {}

    for image_id in pool:
        doc = qdrant.get_documents_by_id([image_id])[0]

        image_thumb = doc.meta.get("thumb_b64")
        image_description = doc.content

        score = judge_candidate(
            query,
            image_thumb,
            image_description,
        )

        relevance[image_id] = score["relevance"]

    # 4. Store everything together
    evaluation.append({
        "query": query,
        "bm25": bm25_ranking,
        "qwen": qwen_ranking,
        "relevance": relevance,
    })

    return evaluation

In [ ]:
evaluation = evaluate_queries(
    QUERIES[0],
    bm25_results,
    qwen_results,
)

evaluation

In [ ]:
def dcg(relevances):
    return sum(
        (2 ** rel - 1) / math.log2(i + 2)
        for i, rel in enumerate(relevances)
    )


def ndcg_at_k(system_ranking, relevance, k=5):
    # Relevance scores in the order returned by the system
    system_relevances = [
        relevance[image_id]
        for image_id in system_ranking[:k]
    ]

    # Ideal ranking comes from the entire common pool
    ideal_relevances = sorted(
        relevance.values(),
        reverse=True
    )[:k]

    ideal_dcg = dcg(ideal_relevances)

    if ideal_dcg == 0:
        return 0.0

    return dcg(system_relevances) / ideal_dcg

In [ ]:
bm25_ndcgs = []
qwen_ndcgs = []

for result in evaluation:

    bm25_score = ndcg_at_k(
        result["bm25"],
        result["relevance"],
        k=TOP_K,
    )

    qwen_score = ndcg_at_k(
        result["qwen"],
        result["relevance"],
        k=TOP_K,
    )

    bm25_ndcgs.append(bm25_score)
    qwen_ndcgs.append(qwen_score)

In [ ]:
print("BM25:", sum(bm25_ndcgs) / len(bm25_ndcgs))
print("InternRAG:", sum(qwen_ndcgs) / len(qwen_ndcgs))